![alt text](pageIndex.jpg)

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import requests


In [2]:
from pageindex import PageIndexClient
import pageindex.utils as utils

# Get your PageIndex API key from https://dash.pageindex.ai/api-keys
PAGEINDEX_API_KEY = os.getenv("pageindex_api_key")
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [3]:
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # any string works
)

async def call_llm(prompt, model="qwen3.5:0.8b", temperature=0):
    response = await client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )

    return response.choices[0].message.content.strip()

### Step 1: PageIndex Tree Generation
##### 1.1 Submit a document for generating PageIndex tree

In [6]:

# You can also use our GitHub repo to generate PageIndex tree
# https://github.com/VectifyAI/PageIndex

# pdf_url = "https://arxiv.org/pdf/2501.12948.pdf"
# pdf_path = os.path.join("data", pdf_url.split('/')[-1])
# os.makedirs(os.path.dirname(pdf_path), exist_ok=True)

# response = requests.get(pdf_url)
# with open(pdf_path, "wb") as f:
#     f.write(response.content)
# print(f"Downloaded {pdf_url}")
pdf_path = "/Users/abhisheksingh/Desktop/vector_less_RAG/data/entropy-22-00193.pdf"

doc_id = pi_client.submit_document(pdf_path)["doc_id"]
print('Document Submitted:', doc_id)

Document Submitted: pi-cmn8hr1r0007z0bphbw0fvjj1


##### 1.2 Get the generated PageIndex tree structure

In [4]:
docs = pi_client.list_documents()

In [5]:
docs

{'documents': [{'id': 'pi-cmn8hr1r0007z0bphbw0fvjj1',
   'name': 'entropy-22-00193.pdf',
   'description': 'This paper introduces a modified deep residual neural network architecture for nonlinear regression, detailing its methodology, experimental evaluation on simulated and real-world climate data, and comparisons with existing approximation techniques.',
   'status': 'completed',
   'createdAt': '2026-03-27T05:59:08.755000',
   'pageNum': 14,
   'folderId': None}],
 'total': 1,
 'limit': 50,
 'offset': 0}

In [7]:
doc_id = "pi-cmn8hr1r0007z0bphbw0fvjj1"

In [8]:
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    print('Simplified Tree Structure of the Document:')
    utils.print_tree(tree)
else:
    print("Processing document, please try again later...")

Simplified Tree Structure of the Document:
[{'title': 'Deep Residual Learning for Nonlinear Reg...',
  'node_id': '0000',
  'prefix_summary': 'This paper presents a deep residual neur...',
  'nodes': [{'title': '1. Introduction',
             'node_id': '0001',
             'summary': 'The text discusses the importance and ch...'},
            {'title': '2. Methodology',
             'node_id': '0002',
             'prefix_summary': '## 2. Methodology\n',
             'nodes': [{'title': '2.1. Architecture of Deep Regression Mod...',
                        'node_id': '0003',
                        'summary': 'The text introduces Residual Neural Netw...'},
                       {'title': '2.2. Datasets',
                        'node_id': '0004',
                        'summary': 'This section introduces nonlinear functi...'}]},
            {'title': '3. Results',
             'node_id': '0005',
             'prefix_summary': '## 3. Results\n',
             'nodes': [{'title': '3.1.

### Step 2: Reasoning-Based Retrieval with Tree Search
##### 2.1 Use LLM for tree search and identify nodes that might contain relevant context

In [9]:
import json

query = "What are the conclusions in this document?"

tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = await call_llm(search_prompt)

##### 2.2 Print retrieved nodes and reasoning process

In [10]:
node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The question asks for the conclusions in the document. I need to scan the document tree structure to
find the section explicitly labeled 'Conclusions'. In the provided tree, Node 0010 is titled '4.
Conclusions'. This section summarizes the main findings, recommendations, and future work of the
paper. Therefore, Node 0010 is the relevant node.

Retrieved Nodes:
Node ID: 0010	 Page: 11	 Title: 4. Conclusions


#### Step 3: Answer Generation
##### 3.1 Extract relevant context from retrieved nodes

In [11]:
node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

## 4. Conclusions

In this paper, we develop deep residual regression models for nonlinear regression. The traditional
deep residual learning behaves well in the images process due to local convolution kernels and deep
neural networks. However, the convolution kernels have no effects on the whole input sequence.
Therefore, it is not suitable for the regression of nonlinear functions. We replace convolutional
layers and pooling layers by fully connected layers to ensure that deep residual learning can be
applied

in nonlinear regression. The residual regression model is carefully and numerically evaluated on
simulated nonlinear data, and the results show that the improved regression model works well. It is
recommended to avoid setting the residual regression model into a great depth or a small width,
since it has a great testing loss under these circumstances. In addition, we compare the residual
regression model with other linear and nonlinear approximation techniqu

##### 3.2 Generate answer based on retrieved context

In [ ]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await call_llm(answer_prompt)
utils.print_wrapped(answer)

Generated Answer:

